<a href="https://colab.research.google.com/github/google-ai-edge/litert-samples/blob/main/benchmark/developer_device_platform/ddp_cli_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2026 The AI Edge Authors.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# ==============================================================================

# Benchmarking on DDP with the `device-run` CLI

> Just need a number for one model on one device? [`litert_cli_benchmark.ipynb`](litert_cli_benchmark.ipynb) does that with a single `litert benchmark --ddp` command.

This notebook calls the [Developer Device Platform](https://docs.cloud.google.com/developer-device-platform/overview) (DDP) CLI, `gcloud alpha device-run`, directly. That gives you:

* **Any runtime.** It uses [LiteRT-LM](https://github.com/google-ai-edge/LiteRT-LM) as the example, but DDP runs whatever Android binary you hand it. Benchmarking another runtime means changing the executable and its flags.
* **Lower-level control.** You choose every flag, and you get back the files the binary writes (here, a metrics protobuf) along with each device's `logcat.txt`.
* **Many devices at once.** One command fans the same run out across a list of devices.

The example benchmarks **Gemma 4 E2B** with [`litert_lm_advanced_main`](https://github.com/google-ai-edge/LiteRT-LM/blob/main/runtime/engine/litert_lm_advanced_main.cc) in two ways:

* **Part 1: Direct call.** The binary itself is the DDP executable. One configuration (for example, GPU) across several devices.
* **Part 2: Wrapper script.** A shell script is the executable and runs several configurations (CPU, then GPU) on each device, with a warm-up pass and a cooldown between them.


## 🛠️ 1. Environment setup

In [ ]:
# @title Set up your GCP project for DDP
# @markdown Set your GCP project id (DDP sessions are billed to it). This cell
# @markdown logs in to gcloud and enables the Device Run API.

ddp_gcp_project = "your-own-gcp-project-id" # @param {type:"string"}

# @markdown The GCS bucket DDP uses for inputs and results. Leave blank to use
# @markdown the default `{project_id}-devicerun`.
custom_bucket_id = "" # @param {type:"string"}

bucket_id = custom_bucket_id if custom_bucket_id else f"{ddp_gcp_project}-devicerun"

if not ddp_gcp_project or ddp_gcp_project == "your-own-gcp-project-id":
  raise ValueError("Please specify a valid GCP Project ID in the form above.")

# Login to gcloud and set Application Default Credentials via Colab Auth
from google.colab import auth
auth.authenticate_user()

# Enable the devicerun API
!gcloud services enable devicerun.googleapis.com --project {ddp_gcp_project}

# DDP stages inputs and writes results here; create it if it is missing.
!gcloud storage buckets describe "gs://{bucket_id}" >/dev/null 2>&1 \
  || gcloud storage buckets create "gs://{bucket_id}" --project={ddp_gcp_project}

print(f"Project: {ddp_gcp_project}\nBucket:  gs://{bucket_id}")


In [ ]:
# @title Download latest version of GCloud CLI
# @markdown This ensures the latest features from DDP CLI are available in the colab.

import os

# 1. Download and extract the absolute latest gcloud release directly to /opt
!wget -qO- https://dl.google.com/dl/cloudsdk/channels/rapid/downloads/google-cloud-cli-linux-x86_64.tar.gz | tar xz -C /opt

# 2. Run the silent install script
!/opt/google-cloud-sdk/install.sh -q

# 3. Add the new installation to the front of the Python environment's PATH
os.environ["PATH"] = f"/opt/google-cloud-sdk/bin:{os.environ['PATH']}"

# 4. Explicitly install the alpha components using the newly downloaded CLI
!gcloud components install alpha --quiet


## 📦 2. Binaries and Model

**The Binaries** are hosted in the public LiteRT release bucket on Google Cloud Storage under `gs://litert/binaries/latest/android_arm64/litert_lm/`.

To run the benchmark on a physical Android device, we need the main executable along with its specific shared library (`.so`) dependencies:

| File | Purpose |
| :--- | :--- |
| `litert_lm_advanced_main` | The core benchmark executable. |
| `libGemmaModelConstraintProvider.so` | **Required.** A link-time (`DT_NEEDED`) dependency for Gemma models. |
| `libLiteRtOpenClAccelerator.so` | **GPU only.** The OpenCL delegate library, dynamically loaded at runtime when `--backend=gpu` is passed. |
| `libLiteRtTopKOpenClSampler.so` | **GPU only.** The OpenCL sampler library, also dynamically loaded for GPU inference. |


**A note on execution:** Because the two OpenCL libraries are opened at *runtime* (`dlopen`) rather than linked at compile time, the Android loader must be explicitly told where to find them. This is why both parts below set `LD_LIBRARY_PATH=/data/local/tmp`.

**The Model**: For this colab, we use [Gemma4 E2B](https://huggingface.co/litert-community/gemma-4-E2B-it-litert-lm). It is pulled straight from Hugging Face.


In [ ]:
# @title Stage the model in your GCS bucket
# @markdown Downloads the model from HuggingFace and copies it to your bucket.
# @markdown DDP pushes files to devices from GCS, so it has to live there first.
# @markdown
# @markdown This is a ~2.6 GB transfer on the first run. The cell skips the work
# @markdown if the object is already in the bucket.
import subprocess

# Gemma4 E2B model on HF
HF_REPO = "litert-community/gemma-4-E2B-it-litert-lm"
HF_FILE = "gemma-4-E2B-it.litertlm"
# Model location in GCS
MODEL_URI = f"gs://{bucket_id}/models/{HF_FILE}"

# Check if the model already exists, to avoid a 2.6 GB download + upload cycle.
exists = subprocess.run(["gcloud", "storage", "ls", MODEL_URI],
                        capture_output=True).returncode == 0

if exists:
  print(f"Already staged, skipping download: {MODEL_URI}")
else:
  url = f"https://huggingface.co/{HF_REPO}/resolve/main/{HF_FILE}"
  print(f"Downloading {url}")
  !curl -L --fail -o "{HF_FILE}" "{url}"
  !gcloud storage cp "{HF_FILE}" "{MODEL_URI}"
  # Free the Colab disk; the copy in GCS is the one that matters.
  !rm -f "{HF_FILE}"

print(f"Model: {MODEL_URI}")

# Public LiteRT release bucket. Use the copy inside `litert_lm/`: the binary one
# directory up is an older build and does not match these libraries.
LITERT_LM = "gs://litert/binaries/latest/android_arm64/litert_lm"
LM_BIN = f"{LITERT_LM}/litert_lm_advanced_main"

WORK = "/data/local/tmp"  # where DDP pushes files on the device

# Libraries that must sit next to the binary on the device.
DEVICE_LIBS = [
    "libGemmaModelConstraintProvider.so",
    "libLiteRtOpenClAccelerator.so",
    "libLiteRtTopKOpenClSampler.so",
]
LIB_PUSH = [f"{LITERT_LM}/{lib}={WORK}/{lib}" for lib in DEVICE_LIBS]


## 📱 3. Choose devices

Both parts refer to devices by their DDP device ID, such as `pa3q-35` or `caiman-35`. Browse the [device catalog](https://docs.cloud.google.com/developer-device-platform/device-catalog) to see what is available, or list it from a scratch cell:

```bash
!gcloud alpha device-run devices list --project="{ddp_gcp_project}" --location="global"
```


## 🎯 Part 1: The direct call

The simplest way to use DDP: hand it the benchmark binary, run it on some devices, and pull back what it writes.

```bash
gcloud alpha device-run sessions submit android-executable \
  --device="pa3q-35,m2q-36,caiman-35" \
  --executable=gs://.../litert_lm_advanced_main \
  --executable-args="--model_path=...,--backend=gpu,--metric_proto_file_path=/data/local/tmp/metrics.pb,..." \
  --executable-env-vars="LD_LIBRARY_PATH=/data/local/tmp" \
  --other-files-to-push="...=/data/local/tmp/model.litertlm" \
  --paths-to-pull="/data/local/tmp/metrics.pb"
```

* **`--device`**: a comma-separated list. DDP runs one job per device.
* **`--executable`**: the compiled binary itself. No shell script.
* **`--executable-args`**: a *comma-separated* list of flags passed to the binary. If a value itself contains a comma, escape it with the `^SEP^` syntax.
* **`--executable-env-vars`**: there is no shell to `export LD_LIBRARY_PATH`, so it is set here. GPU runs need it so the binary can load the OpenCL delegate pushed next to it.
* **`--paths-to-pull`**: files to copy back to your bucket when the job ends.

### 📝 Where the results come from
Given `--metric_proto_file_path`, `litert_lm_advanced_main` writes its benchmark metrics to a protobuf file, and `--paths-to-pull` brings that file back. The numbers below are read straight from it.

DDP also uploads each device's `logcat.txt`, which contains the binary's console output. It is used for one check only: confirming that the GPU delegate (`LITERT_CL`) actually loaded, because the metrics file does not record which backend ran.

DDP names each device's results `job-000`, `job-001`, … in the order the devices were listed, which is how the results are labeled.

### ⚖️ The trade-off
`--executable-args` applies to the whole session, so every device runs **the same configuration**. To compare CPU *and* GPU you would submit two sessions, or use a wrapper script as in Part 2.


In [ ]:
# @title Run one configuration directly on several devices
# @markdown No shell script: the binary is the DDP executable.

DIRECT_DEVICES = "pa3q-35,m2q-36,caiman-35" # @param {type:"string"}
DIRECT_BACKEND = "gpu" # @param ["cpu", "gpu"]

PREFILL_TOKENS = 1024 # @param {type:"integer"}
DECODE_TOKENS  = 512 # @param {type:"integer"}
MAX_NUM_TOKENS = 2048 # @param {type:"integer"}
CPU_THREADS    = 4 # @param {type:"integer"}

# LM_BIN, LIB_PUSH, WORK and MODEL_URI all come from the setup section above.
METRICS_PB = f"{WORK}/metrics.pb"
args = [
    f"--model_path={WORK}/model.litertlm",
    "--benchmark=true",
    f"--benchmark_prefill_tokens={PREFILL_TOKENS}",
    f"--benchmark_decode_tokens={DECODE_TOKENS}",
    f"--max_num_tokens={MAX_NUM_TOKENS}",
    "--report_peak_memory_footprint=true",
    f"--backend={DIRECT_BACKEND}",
    # The binary writes its metrics here; --paths-to-pull brings the file back.
    f"--metric_proto_file_path={METRICS_PB}",
]
# On GPU the driver picks its own thread count, so the flag is CPU-only.
if DIRECT_BACKEND == "cpu":
  args.append(f"--num_cpu_threads={CPU_THREADS}")

push = [f"{MODEL_URI}={WORK}/model.litertlm"] + LIB_PUSH

direct_output = !gcloud alpha device-run sessions submit android-executable \
  --project="{ddp_gcp_project}" \
  --location="global" \
  --bucket-name="{bucket_id}" \
  --device="{DIRECT_DEVICES}" \
  --executable="{LM_BIN}" \
  --executable-args="{','.join(args)}" \
  --executable-env-vars="LD_LIBRARY_PATH={WORK}" \
  --other-files-to-push="{','.join(push)}" \
  --paths-to-pull="{METRICS_PB}" \
  --executable-timeout=30m 2>&1

print("\n".join(direct_output))


In [ ]:
# @title Helpers: read the metrics files the binary writes
import re
from pathlib import Path

import pandas as pd

# --- Metrics schema -----------------------------------------------------------
# --metric_proto_file_path writes a binary protobuf. No PyPI package ships its
# schema, so fetch the two public .proto files from LiteRT-LM, compile them to a
# descriptor set, and build the message class at runtime. A descriptor set,
# rather than generated _pb2.py modules, keeps this independent of whichever
# protobuf version the runtime has installed.
!pip install -q grpcio-tools
PROTO_DIR = "/content/litert_lm_proto"
PROTO_RAW = "https://raw.githubusercontent.com/google-ai-edge/LiteRT-LM/main"
!mkdir -p {PROTO_DIR}/runtime/proto
for _p in ["engine.proto", "litert_lm_metrics.proto"]:
  !curl -sfL {PROTO_RAW}/runtime/proto/{_p} -o {PROTO_DIR}/runtime/proto/{_p}
!python -m grpc_tools.protoc -I{PROTO_DIR} --include_imports \
    --descriptor_set_out={PROTO_DIR}/metrics.desc \
    {PROTO_DIR}/runtime/proto/engine.proto \
    {PROTO_DIR}/runtime/proto/litert_lm_metrics.proto

from google.protobuf import descriptor_pb2, descriptor_pool, message_factory
_pool = descriptor_pool.DescriptorPool()
for _f in descriptor_pb2.FileDescriptorSet.FromString(
    Path(f"{PROTO_DIR}/metrics.desc").read_bytes()).file:
  _pool.Add(_f)
MetricsList = message_factory.GetMessageClass(
    _pool.FindMessageTypeByName("litert.lm.proto.LitertLmMetricsList"))

# --- Reading results ----------------------------------------------------------
# Session artifacts are downloaded here, and every reader below looks here.
RESULTS_DIR = "/content/ddp_results"

METRIC_KEYS = ["prefill_tok_s", "decode_tok_s", "ttft_s", "peak_ram_mb"]


def read_metrics(pb_path) -> dict:
    """Returns prefill/decode tokens/s, TTFT and peak RAM from a metrics file.

    A failed run writes no file, so a missing file yields empty values.
    """
    if pb_path is None or not Path(pb_path).exists():
        return dict.fromkeys(METRIC_KEYS)
    runs = MetricsList.FromString(Path(pb_path).read_bytes()).metrics
    if not runs:
        return dict.fromkeys(METRIC_KEYS)
    m = runs[-1]
    return {
        "prefill_tok_s": m.prefill_turns[-1].tokens_per_second if m.prefill_turns else None,
        "decode_tok_s": m.decode_turns[-1].tokens_per_second if m.decode_turns else None,
        "ttft_s": m.time_to_first_token_seconds,
        "peak_ram_mb": m.peak_mem_mb,
    }


def gpu_delegated(text_path) -> bool:
    """True if the OpenCL delegate loaded. The metrics file does not record this."""
    return (text_path is not None and Path(text_path).exists()
            and "LITERT_CL" in Path(text_path).read_text(errors="replace"))


def get_session_id(output_lines):
    """Extracts the session ID from the output of `sessions submit`."""
    for line in output_lines:
        m = re.search(r"Creating session \[([^\]]+)\]", line)
        if m:
            return m.group(1)
    return None


def download_session_results(session_id):
    """Copies a session's artifacts from GCS into RESULTS_DIR.

    Assumes `bucket_id` is defined as a global variable in the notebook.
    """
    !mkdir -p {RESULTS_DIR}
    !gcloud storage cp -r gs://{bucket_id}/automation/sessions/{session_id}/ {RESULTS_DIR}/


def direct_results(session_id, devices, backend) -> pd.DataFrame:
    """One row per device for a Part 1 session.

    DDP names jobs job-000, job-001, ... in the order the devices were passed to
    --device, and nothing inside a job records the device, so rows are labeled
    by that order.
    """
    jobs = sorted(Path(f"{RESULTS_DIR}/{session_id}").glob("job-*"))
    if len(jobs) != len(devices):
        print(f"Warning: {len(devices)} devices requested, "
              f"{len(jobs)} job folders found.")
    rows = []
    for device, job in zip(devices, jobs):
        row = {"device": device, "job": job.name,
               **read_metrics(next(job.rglob("metrics.pb"), None))}
        if backend == "gpu":
            row["gpu_ok"] = gpu_delegated(next(job.rglob("logcat.txt"), None))
        rows.append(row)
    return pd.DataFrame(rows).set_index("device")


def generate_dashboard(session_id: str) -> pd.DataFrame:
    """Tabulates a Part 2 session and plots CPU vs GPU for each device."""
    local_dir = Path(f"{RESULTS_DIR}/{session_id}")
    rows = []

    # The script writes <tag>.exit for every configuration, even failed ones,
    # so iterating over those means a failed run still gets a row.
    for exit_file in sorted(local_dir.rglob("out/*.exit")):
        out, tag = exit_file.parent, exit_file.stem  # tag is 'cpu' or 'gpu'
        dev_file = out / "_device.txt"
        device = (dev_file.read_text().strip() if dev_file.exists()
                  else re.search(r"job-\d+", str(out)).group())
        rows.append({
            "device": device,
            "backend": tag,
            "exit": exit_file.read_text().strip(),
            **read_metrics(out / f"{tag}.pb"),
            "gpu_ok": gpu_delegated(out / f"{tag}.log") if tag == "gpu" else None,
        })

    if not rows:
        print(f"No results found under {local_dir} -- did the download succeed?")
        return None

    df = pd.DataFrame(rows).set_index(["device", "backend"]).sort_index()
    print(df.to_string(float_format=lambda v: f"{v:,.1f}"))

    # Alert on failures or missing GPU delegation.
    bad = df[(df.exit != "0") | (df.gpu_ok == False)]  # noqa: E712
    if not bad.empty:
        print("\nFAILED or no GPU delegation: "
              + ", ".join(f"{d}/{b}" for d, b in bad.index))

    for metric in METRIC_KEYS:
        df[metric].unstack("device").plot.bar(title=metric, figsize=(8, 3), rot=0)

    return df


In [ ]:
# @title Show the direct run's results
# @markdown Downloads each device's results and reads the metrics file the
# @markdown binary wrote.

direct_session = get_session_id(direct_output)
if not direct_session:
    raise SystemExit("Could not find a session ID; check the output above.")

download_session_results(direct_session)
devices = [d.strip() for d in DIRECT_DEVICES.split(",") if d.strip()]
direct_df = direct_results(direct_session, devices, DIRECT_BACKEND)
print(direct_df.to_string(float_format=lambda v: f"{v:,.1f}"))


## 🔁 Part 2: The wrapper script

The direct call runs one configuration. To compare **CPU against GPU** on each device, hand DDP a shell script instead of the binary and let the script run the binary several times.

That buys three things a single invocation cannot:

* **Warm-start measurement.** Each configuration runs *twice*. The first pass is discarded; it only writes the XNNPACK / ML Drift caches next to the model. The second pass is measured and writes the metrics file.
* **Thermal cooldown.** A `sleep 20` between configurations stops one run from throttling the next.
* **Device identification.** The script records `getprop ro.product.model`, so results carry the real model name instead of relying on submission order.

For each configuration the script writes a metrics file, the console output and the exit code into `/data/local/tmp/out`, and `--paths-to-pull` brings that folder back.


In [ ]:
%%writefile run_all.sh
#!/system/bin/sh

WORK="/data/local/tmp"
# The OpenCL delegate is loaded at runtime, so it has to be on the loader path.
export LD_LIBRARY_PATH="$WORK"
chmod 755 "$WORK/litert_lm_advanced_main"
rm -rf "$WORK/out" && mkdir -p "$WORK/out"

# DDP reports only job-NNN. Without this the report cannot name the hardware.
getprop ro.product.model > "$WORK/out/_device.txt"

COMMON="--model_path=$WORK/model.litertlm \
    --benchmark=true \
    --benchmark_prefill_tokens=1024 \
    --benchmark_decode_tokens=512 \
    --max_num_tokens=2048 \
    --report_peak_memory_footprint=true"

# run <tag> <extra flags...>
#   First pass warms the caches and is discarded; second pass is measured.
run() {
  tag="$1"; shift
  echo "### $tag"
  "$WORK/litert_lm_advanced_main" $COMMON "$@" >/dev/null 2>&1
  "$WORK/litert_lm_advanced_main" $COMMON "$@" \
      --metric_proto_file_path="$WORK/out/$tag.pb" \
      >"$WORK/out/$tag.log" 2>&1
  echo $? > "$WORK/out/$tag.exit"
  sleep 20   # let the SoC cool so the next config is not throttled
}

run cpu --backend=cpu --num_cpu_threads=4
run gpu --backend=gpu

echo "ALL DONE"


In [ ]:
# @title Upload the script and submit the sweep
# @markdown Submits one job per selected device, each running the full
# @markdown CPU + GPU sweep. Uses `--async` so the next cell can report progress.
import uuid

# @markdown The devices to run DDP CLI on;
ddp_target_devices = "pa3q-35,m2q-36,caiman-35" # @param {type: "string"}

# A unique path per run: overwriting a shared script would corrupt any session
# that has not started yet, because DDP fetches the script when the job starts.
run_tag = uuid.uuid4().hex[:8]
SCRIPT_URI = f"gs://{bucket_id}/scripts/{run_tag}/run_all.sh"
!gcloud storage cp run_all.sh "{SCRIPT_URI}"

push = [
    f"{MODEL_URI}={WORK}/model.litertlm",
    f"{LM_BIN}={WORK}/litert_lm_advanced_main",
] + LIB_PUSH

submit_output = !gcloud alpha device-run sessions submit android-executable \
  --project="{ddp_gcp_project}" \
  --location="global" \
  --bucket-name="{bucket_id}" \
  --device="{ddp_target_devices}" \
  --executable="{SCRIPT_URI}" \
  --other-files-to-push="{','.join(push)}" \
  --paths-to-pull="{WORK}/out" \
  --executable-timeout=60m \
  --async 2>&1

print("\n".join(submit_output))


In [ ]:
# @title Poll until the sweep completes, then download the results
# @markdown `sessions wait` blocks silently, so we poll `sessions describe`
# @markdown instead and print progress as jobs finish.
# @markdown
# @markdown ---
# @markdown **Polling interval (seconds):**
import re
import subprocess
import time

POLL_SECONDS = 60 # @param {type:"integer"}

session_id = get_session_id(submit_output)
if not session_id:
    raise SystemExit("Failed to find session ID in the output.")

print(f"Waiting for {session_id}, polling every {POLL_SECONDS}s...\n")

start = time.time()
while True:
    out = subprocess.run(
        ["gcloud", "alpha", "device-run", "sessions", "describe", session_id,
         f"--project={ddp_gcp_project}", "--location=global"],
        capture_output=True, text=True)
    # `describe` writes its status prose to stderr and the job table to stdout.
    status = (out.stderr + out.stdout).strip()

    # Keep just the status and the job tally; drop the long bucket URL.
    keep = [l for l in status.splitlines()
            if l.startswith("Session [") or l.startswith("Job status:")]
    print(f"[{(time.time() - start) / 60:5.1f} min] " + " | ".join(keep))
    if out.returncode != 0:
        print(f"  (describe exited {out.returncode})")

    if "finished with result" in status:
        break
    time.sleep(POLL_SECONDS)


In [ ]:
# @title 📊 Results dashboard
# @markdown Reads each device's metrics files into a table and plots CPU vs GPU.
download_session_results(session_id)
generate_dashboard(session_id)